In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from tqdm import tqdm

path_seq = (
  'klines_1m\\um_futures\\btcusdt_perpetual.csv',
  'klines_1m\\um_futures\\ethusdt_perpetual.csv',
  'klines_1m\\um_futures\\solusdt_perpetual.csv',
  'klines_1m\\um_futures\\bnbusdt_perpetual.csv'
)

columns = pd.Index((
  'Open', 'High', 'Low', 'Close', 'Volume', 'Quote asset volume',
  'Number of trades', 'Taker buy volume', 'Taker buy QAV'
))

idx_col = 'Open time'
ohlc_cols = pd.Index(('Open', 'High', 'Low', 'Close'))
price_cols = pd.Index(('Open', 'Close', 'Buy avg', 'Sell avg', 'Average'))

index = pd.date_range(
  '2021-07-01 00:00:00',
  '2026-07-01 00:00:00',
  freq='min',
  inclusive='left',
  name=idx_col
)

n_assets = len(path_seq)
n_obs = len(index)

print(f'Assets: {n_assets}')
print(f'Observations: {n_obs}\n')

eps = 1e-6

df_list = []
for path in path_seq:
  df = pd.read_csv(path)
  df[idx_col] = pd.to_datetime(df[idx_col])
  df.set_index(idx_col, inplace=True)
  assert df.columns.equals(columns)
  assert df.index.equals(index)

  df['Taker sell volume'] = df['Volume'] - df['Taker buy volume']
  df['Taker sell QAV'] = df['Quote asset volume'] - df['Taker buy QAV']
  df['Buy avg'] = df['Taker buy QAV'] / df['Taker buy volume']
  df['Sell avg'] = df['Taker sell QAV'] / df['Taker sell volume']
  df['Average'] = df['Quote asset volume'] / df['Volume']

  mask = ~(
    np.isfinite(df).all(axis=1) &
    df.gt(0.0, 'index').all(axis=1) &
    df[price_cols].gt(df['Low'] - eps, 'index').all(axis=1) &
    df[price_cols].lt(df['High'] + eps, 'index').all(axis=1)
  )
  n_gaps = mask.sum()
  gap_pct = n_gaps / n_obs * 100.0
  print(f'{path}: {n_gaps} gaps found ({gap_pct:.2f}%)')

  df = df[ohlc_cols]
  df[mask] = np.nan
  df['Close'] = df['Close'].ffill().bfill()
  df[mask] = df[['Close'] * 4][mask]
  df_list.append(df)

Assets: 4
Observations: 2629440

klines_1m\um_futures\btcusdt_perpetual.csv: 150 gaps found (0.01%)
klines_1m\um_futures\ethusdt_perpetual.csv: 129 gaps found (0.00%)
klines_1m\um_futures\solusdt_perpetual.csv: 165 gaps found (0.01%)
klines_1m\um_futures\bnbusdt_perpetual.csv: 163 gaps found (0.01%)


In [2]:
ohlc = torch.from_numpy(np.stack(df_list, 1))

class Model(nn.Module):
  def __init__(self, d_input: int, d_hidden: int, d_output: int):
    super().__init__()
    self.gru = nn.GRU(d_input, d_hidden, batch_first=True)
    self.fc = nn.Linear(d_hidden, d_output)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.fc(self.gru(x)[1][0])

model = Model(3, 30, 2)
optimizer = optim.NAdam(model.parameters(), 3e-4)

n_params = sum(param.numel() for param in model.parameters())
print(n_params)

3212


In [3]:
min_tf = 60
max_tf = 480
min_frames = 10
max_frames = 30

maker_fee = 0.0002
taker_fee = 0.0005
deposit = 100.0

batch_size = 32
n_batches = 256
n_epochs = 10

for epoch in range(n_epochs):

  model.train()
  total_gain, total_wins = 0.0, 0.0
  pbar = tqdm(range(1, n_batches + 1), f'epoch={epoch} training')
  for n in pbar:

    optimizer.zero_grad()
    pnls = []
    for _ in range(batch_size):

      asset = np.random.choice(n_assets)
      tf = np.random.randint(min_tf, max_tf + 1)
      n_frames = np.random.randint(min_frames, max_frames + 1)
      lookback = (n_frames - 1) * tf + 1
      patience = tf
      t0 = np.random.randint(tf + lookback - 1, n_obs - patience - 1)

      sample = ohlc[t0 + 1 - tf - lookback: t0 + 1, asset, 1:]
      windows = sample[1:].unfold(0, tf, 1)
      x = torch.stack((
        windows[::tf, 0].max(1).values / sample[:-tf:tf, 2],
        windows[::tf, 1].min(1).values / sample[:-tf:tf, 2],
        sample[tf::tf, 2] / sample[:-tf:tf, 2]
      ), 1).log().float()
      sigma = x[:, 2].std()

      base = ohlc[t0, asset, 3]
      qty = deposit / base
      spent, taken = 0.0, 0.0
      bprice, sprice = (model(x / sigma) * sigma).exp() * base

      for t in range(t0 + 1, n_obs):
        span = t - t0 - 1
        o, h, l, c = ohlc[t, asset]
        if span == patience:
          if not spent: spent = bprice.detach() / bprice * o * (1.0 + taker_fee) * qty
          if not taken: taken = sprice.detach() / sprice * o * (1.0 - taker_fee) * qty
        else:
          if not spent and l <= bprice: spent = (
            bprice / bprice.detach() * o * (1.0 + taker_fee)
            if not span and o <= bprice else bprice * (1.0 + maker_fee)
          ) * qty
          if not taken and h >= sprice: taken = (
            sprice / sprice.detach() * o * (1.0 - taker_fee)
            if not span and o >= sprice else sprice * (1.0 - maker_fee)
          ) * qty
        if spent and taken:
          pnls.append(taken - spent)
          break

    pnls = torch.stack(pnls)
    gain = pnls.mean()
    (-gain).backward()
    optimizer.step()

    total_gain += gain.item()
    total_wins += (pnls > 0.0).float().mean().item()
    pbar.set_postfix_str(f'avg_pnl={total_gain/n:.4f}, win_rate={total_wins/n:.4f}')

epoch=9 training: 100%|██████████| 256/256 [02:48<00:00,  1.52it/s, avg_pnl=-0.0673, win_rate=0.4420]
